# AIMO3 Math Solver — Google Colab

**Model:** `Qwen/Qwen2.5-Math-72B-Instruct` (fp8 → H100 80GB) or `7B-Instruct` (A100 40GB / T4)  
**Answer format:** Integer in [0, 99999]  
**Method:** TIR (Tool-Integrated Reasoning) — generate Python → execute → inject output → majority vote

## GPU requirements
| Colab GPU | VRAM | Model |
|-----------|------|-------|
| H100 80GB | 80 GB | `Qwen2.5-Math-72B-Instruct` + fp8 ✓ |
| A100 40GB | 40 GB | `Qwen2.5-Math-7B-Instruct` ✓ |
| T4 16GB   | 16 GB | `Qwen2.5-Math-7B-Instruct` ✓ |

## Phases
1. Setup (Drive mount + model download)
2. Integer answer extraction + majority vote
3. TIR solver
4. Benchmark on AIMO3 reference problems

---
## 1. Setup

In [ ]:
import os

# Disable HuggingFace XET chunked downloader — causes CAS/IO errors on large models.
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["CUDA_MODULE_LOADING"] = "LAZY"

# Pin packages that Colab pre-installs at conflicting versions.
# vLLM needs newer protobuf + opentelemetry; tensorflow/google-adk don't matter here.
!pip install -q -U \
    "protobuf>=5.26.1" \
    "opentelemetry-api>=1.36.0,<1.39.0" \
    "opentelemetry-sdk>=1.36.0,<1.39.0"

!pip install -q vllm transformers>=4.44.0 sympy pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 6.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpc-google-iam-v1 0.14.3 requires protobuf!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.34.0 which is incompatible.
google-cloud-trace 1.18.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.34.0 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.34.0 which is incompatible.
google-cloud-monitoring 2.29.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.34.0 which is incompatible.
google-cloud-functions 1.22.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU — Runtime → Change runtime type → GPU")

GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU:  {GPU_NAME}")
print(f"VRAM: {VRAM_GB:.1f} GB")

# Auto-select model based on available VRAM
if VRAM_GB >= 75:
    MODEL_ID = "Qwen/Qwen2.5-Math-72B-Instruct"
    USE_FP8  = True
    MAX_TOKENS = 3072
    SOLUTIONS_PER_PROBLEM = 32
    print("→ Using 72B model with fp8 quantization (H100 80GB detected)")
else:
    MODEL_ID = "Qwen/Qwen2.5-Math-7B-Instruct"
    USE_FP8  = False
    MAX_TOKENS = 3072
    SOLUTIONS_PER_PROBLEM = 16
    print(f"→ Using 7B model (VRAM {VRAM_GB:.0f} GB < 75 GB needed for 72B)")

os.makedirs("results", exist_ok=True)
print(f"Model: {MODEL_ID}")
print(f"Max tokens: {MAX_TOKENS}  |  Solutions per problem: {SOLUTIONS_PER_PROBLEM}")

GPU:  NVIDIA A100-SXM4-40GB
VRAM: 39.5 GB
→ Using 7B model (VRAM 39 GB < 75 GB needed for 72B)
Model: Qwen/Qwen2.5-Math-7B-Instruct
Max tokens: 3072  |  Solutions per problem: 16


In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

# Downloads from HuggingFace on first run; cached to Drive on subsequent runs.
# 72B: ~144 GB download (bf16 weights), stored as fp8 in VRAM (~72 GB).
# 7B:  ~15 GB download, no quantization needed.
print(f"Loading tokenizer for {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

print(f"Loading model {MODEL_ID} into vLLM ...")
llm = LLM(
    model=MODEL_ID,
    trust_remote_code=True,
    dtype="auto",
    quantization="fp8" if USE_FP8 else None,
    max_model_len=4096,
    gpu_memory_utilization=0.92,
    disable_log_stats=True,
)

print(f"✓ Model loaded: {MODEL_ID}")

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

config.json:   0%|          | 0.00/658 [00:00<?, ?B/s]

The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 03-08 10:23:55 [model.py:531] Resolved architecture: Qwen2ForCausalLM
INFO 03-08 10:23:55 [model.py:1554] Using max model len 4096
INFO 03-08 10:23:55 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-08 10:23:55 [vllm.py:747] Asynchronous scheduling is enabled.


generation_config.json:   0%|          | 0.00/161 [00:00<?, ?B/s]

WARNING 03-08 10:23:56 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 03-08 10:28:49 [llm.py:388] Supported tasks: ['generate']
✓ Model loaded: Qwen/Qwen2.5-Math-7B-Instruct


---
## 2. Answer Extraction & Voting

AIMO3 answers are always integers in [0, 99999]. No LaTeX normalization needed.

We extract integers from `\boxed{}` and fall back to scanning the last lines of the solution.
Majority vote selects the most-agreed-upon integer.

In [ ]:
import re
import subprocess
import sys
from collections import Counter


# ---------------------------------------------------------------------------
# Integer answer extraction
# AIMO3: every answer is an integer in [0, 99999]. No LaTeX normalization needed.
# ---------------------------------------------------------------------------

def extract_integer(text: str) -> int | None:
    """Extract the final integer answer from a solution.

    Priority:
    1. Last \\boxed{N} where N is an integer (handles \\boxed{42}, \\boxed{1234})
    2. Last bare integer on a line that looks like a conclusion
    3. Any integer in [0, 99999] near "answer is" / "= N" patterns
    Returns None if no valid integer found.
    """
    # --- Strategy 1: \boxed{integer} — walk braces to handle \boxed{\boxed{42}} etc.
    boxed_ints = []
    for m in re.finditer(r'\\boxed\{', text):
        start = m.end()
        depth, i = 1, start
        while i < len(text) and depth > 0:
            if text[i] == '{':
                depth += 1
            elif text[i] == '}':
                depth -= 1
            i += 1
        if depth == 0:
            content = text[start:i - 1].strip()
            # Strip any surrounding \text{} or spaces
            content = re.sub(r'\\text\{([^}]*)\}', r'\1', content).strip()
            # Accept pure integers (optionally with leading/trailing whitespace or commas)
            content_clean = content.replace(',', '').replace(' ', '')
            try:
                val = int(content_clean)
                if 0 <= val <= 99999:
                    boxed_ints.append(val)
            except ValueError:
                pass

    if boxed_ints:
        return boxed_ints[-1]   # last boxed answer is the final one

    # --- Strategy 2: "answer is N" / "= N" / "answer: N" patterns
    patterns = [
        r'(?:answer|result|value)\s*(?:is|=|:)\s*(\d{1,5})\b',
        r'=\s*(\d{1,5})\s*$',
        r'\b(\d{1,5})\s*$',
    ]
    for pat in patterns:
        hits = re.findall(pat, text, re.IGNORECASE | re.MULTILINE)
        for h in reversed(hits):
            val = int(h)
            if 0 <= val <= 99999:
                return val

    return None


def majority_vote(answers: list[int | None]) -> int | None:
    """Return the most common integer answer. None entries are ignored."""
    valid = [a for a in answers if a is not None]
    if not valid:
        return None
    return Counter(valid).most_common(1)[0][0]


# ---------------------------------------------------------------------------
# Code execution (for TIR scoring)
# ---------------------------------------------------------------------------

def execute_code_blocks(text: str, timeout: int = 10) -> tuple[int, int, str]:
    """Run python code blocks. Returns (passed, failed, last_output)."""
    blocks = re.findall(r'```python\n(.*?)```', text, re.DOTALL)
    if not blocks:
        return 0, 0, ""
    combined = "\n".join(b.strip() for b in blocks)
    try:
        result = subprocess.run(
            [sys.executable, "-c", combined],
            capture_output=True, text=True, timeout=timeout,
        )
        if result.returncode == 0:
            return len(blocks), 0, result.stdout.strip()
        else:
            return 0, len(blocks), result.stderr.strip().split('\n')[-1][:200]
    except subprocess.TimeoutExpired:
        return 0, len(blocks), "timeout"
    except Exception as e:
        return 0, len(blocks), str(e)[:200]


# ---------------------------------------------------------------------------
# Smoke tests
# ---------------------------------------------------------------------------

_tests = [
    (r"Therefore $\boxed{42}$.",                    42),
    (r"The answer is $\boxed{1234}$.",             1234),
    (r"\boxed{\boxed{99999}}",                    99999),
    (r"We get \boxed{0}.",                             0),
    (r"The remainder is \boxed{3}. Note: not 7.",      3),  # last boxed wins
    (r"So the value is 567.",                        567),
    (r"answer is 100000",                           None),  # out of range
    (r"No answer here.",                            None),
]

all_ok = True
for text, expected in _tests:
    got = extract_integer(text)
    ok = got == expected
    if not ok:
        all_ok = False
    print(f"  [{'OK' if ok else 'FAIL'}] extract_integer({text[:40]!r}) → {got!r}  (expected {expected!r})")

print(f"\n{'All tests passed!' if all_ok else 'Some tests FAILED.'}")
print("Answer extraction ready.")

  [OK] extract_integer('Therefore $\\boxed{42}$.') → 42  (expected 42)
  [OK] extract_integer('The answer is $\\boxed{1234}$.') → 1234  (expected 1234)
  [OK] extract_integer('\\boxed{\\boxed{99999}}') → 99999  (expected 99999)
  [OK] extract_integer('We get \\boxed{0}.') → 0  (expected 0)
  [OK] extract_integer('The remainder is \\boxed{3}. Note: not 7.') → 3  (expected 3)
  [OK] extract_integer('So the value is 567.') → 567  (expected 567)
  [OK] extract_integer('answer is 100000') → None  (expected None)
  [OK] extract_integer('No answer here.') → None  (expected None)

All tests passed!
Answer extraction ready.


---
## 3. TIR Solver

Qwen2.5-Math-Instruct is specifically trained for Tool-Integrated Reasoning:
it writes Python, pauses at `\n```output`, expects the execution result injected,
then continues until it reaches `\boxed{integer}`.

We run N solutions in parallel (vLLM batching), each with up to `max_rounds` of
code→execute→inject→continue. Final answer = majority vote over extracted integers.

In [ ]:
import time

# Qwen2.5-Math-Instruct TIR system prompt (from the model card / AIMO2 winning setup)
SYSTEM_PROMPT = (
    "Please integrate natural language reasoning with programs to solve the problem above, "
    "and put your final integer answer within \\boxed{}."
)

_STOP_TIR  = ["```output"]     # model pauses here expecting code output injection
_STOP_DONE = ["<|im_end|>", "<|endoftext|>"]


def _build_initial_prompt(problem: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": problem},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def _run_code_block(text: str) -> str:
    """Execute the last python block in text, return output string (or error)."""
    blocks = re.findall(r'```python\n(.*?)```', text, re.DOTALL)
    if not blocks:
        return ""
    combined = "\n".join(b.strip() for b in blocks)
    try:
        res = subprocess.run(
            [sys.executable, "-c", combined],
            capture_output=True, text=True, timeout=10,
        )
        out = res.stdout.strip() if res.returncode == 0 else res.stderr.strip().split("\n")[-1][:300]
        return out or "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: execution timed out"
    except Exception as e:
        return f"Error: {e}"


def solve(problem: str, n: int = SOLUTIONS_PER_PROBLEM, max_rounds: int = 5) -> tuple[int | None, list[int | None]]:
    """Solve a problem using TIR with majority vote.

    Round 0: generate N drafts in parallel, stopping at ```output (code block)
    Rounds 1+: for each draft still running, inject code result then continue
    Final: extract integer from each draft, return majority vote
    """
    base_prompt = _build_initial_prompt(problem)

    # Each state: {"prompt": str, "text": str, "done": bool}
    states = [{"prompt": base_prompt, "text": "", "done": False} for _ in range(n)]

    # ── Round 0: generate all N in parallel ──────────────────────────────────
    params = SamplingParams(
        temperature=0.7, top_p=0.95,
        max_tokens=MAX_TOKENS, n=n,
        stop=_STOP_TIR + _STOP_DONE,
    )
    outputs = llm.generate([base_prompt], params)[0]

    for i, out in enumerate(outputs.outputs):
        states[i]["text"] = out.text
        # Done if stopped at a DONE token (not at ```output)
        stopped_at_output = out.text.rstrip().endswith("```") or \
                            not any(out.text.rstrip().endswith(s.strip()) for s in _STOP_DONE)
        states[i]["done"] = extract_integer(out.text) is not None and not stopped_at_output

    # ── Rounds 1…max_rounds: inject code output and continue ─────────────────
    for rnd in range(1, max_rounds):
        active = [s for s in states if not s["done"]]
        if not active:
            break

        # Inject code output into each active state's prompt
        for s in active:
            code_out = _run_code_block(s["text"])
            # Qwen2.5-Math TIR format: after stopping at ```output, inject result then close block
            continuation = s["prompt"] + s["text"] + "```output\n" + code_out + "\n```\n"
            s["prompt"] = continuation
            s["text"] = ""   # reset; we'll append new generation

        prompts = [s["prompt"] for s in active]
        params_cont = SamplingParams(
            temperature=0.6, top_p=0.95,
            max_tokens=MAX_TOKENS,
            n=1,
            stop=_STOP_TIR + _STOP_DONE,
        )
        cont_outputs = llm.generate(prompts, params_cont)

        for s, out in zip(active, cont_outputs):
            s["text"] = out.outputs[0].text
            # Reconstruct full solution from prompt chain
            full_text = s["prompt"] + s["text"]
            if extract_integer(full_text) is not None:
                s["done"] = True
            s["full_text"] = full_text  # keep for extraction

    # ── Extract integer from each state ──────────────────────────────────────
    answers = []
    for s in states:
        full = s.get("full_text", s["prompt"] + s["text"])
        answers.append(extract_integer(full))

    return majority_vote(answers), answers


# Quick smoke test
t0 = time.time()
ans, all_ans = solve("What is $7^2 - 3^2$?", n=4, max_rounds=2)
print(f"Answer: {ans}  (expected 40)")
print(f"All answers: {all_ans}")
print(f"Time: {time.time()-t0:.1f}s")

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/4 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Answer: 40  (expected 40)
All answers: [40, 40, 40, 40]
Time: 2.7s


---
## 4. Benchmark on Reference Problems

AIMO3 provides 10 reference problems with known answers.
We run our solver on them to verify correctness before submission.

In [ ]:
import pandas as pd
import json

# ── Load AIMO3 reference problems ────────────────────────────────────────────
# Option A (Colab): upload reference.csv manually via the Files panel, or
#                   place it in Google Drive and set the path below.
# Option B: download it from the AIMO3 Kaggle competition data page and upload.
#
# The file has columns: id, problem, answer

REF_PATH = "reference.csv"   # adjust if stored elsewhere in Drive

try:
    ref_df = pd.read_csv(REF_PATH)
    print(f"Loaded {len(ref_df)} reference problems from {REF_PATH}")
    print(ref_df[["id", "answer"]].to_string())
except FileNotFoundError:
    print(f"'{REF_PATH}' not found.")
    print("Upload reference.csv via the Colab Files panel (folder icon on the left),")
    print("or set REF_PATH to its Google Drive location.")
    ref_df = None

'reference.csv' not found.
Upload reference.csv via the Colab Files panel (folder icon on the left),
or set REF_PATH to its Google Drive location.


In [ ]:
if ref_df is not None:
    ref_results = []
    t0 = time.time()

    for _, row in ref_df.iterrows():
        pred, all_preds = solve(row["problem"])
        correct = (pred == int(row["answer"]))
        ref_results.append({
            "id":        row["id"],
            "expected":  int(row["answer"]),
            "predicted": pred,
            "correct":   correct,
            "all":       all_preds,
        })
        votes = Counter(a for a in all_preds if a is not None)
        status = "CORRECT" if correct else "WRONG"
        print(f"[{status}] id={row['id']}  expected={row['answer']}  got={pred}  votes={dict(votes.most_common(3))}")

    elapsed = time.time() - t0
    accuracy = sum(r["correct"] for r in ref_results) / len(ref_results)
    print(f"\nReference accuracy: {accuracy:.0%} ({sum(r['correct'] for r in ref_results)}/{len(ref_results)})  in {elapsed:.1f}s")
    print(f"Avg time per problem: {elapsed/len(ref_results):.1f}s")

    with open("results/reference_benchmark.json", "w") as f:
        json.dump(ref_results, f, indent=2, default=str)
    print("Saved to results/reference_benchmark.json")
else:
    print("Skipped — no reference file.")

Skipped — no reference file.


---
## 5. Run on a Test CSV

Upload `test.csv` (from the AIMO3 Data page) via the Colab Files panel,
then run the cell below to produce `submission.csv`.

`test.csv` columns: `id`, `problem`  
`submission.csv` columns: `id`, `answer`

In [ ]:
TEST_PATH = "test.csv"        # upload via Colab Files panel
OUT_PATH  = "submission.csv"

try:
    test_df = pd.read_csv(TEST_PATH)
except FileNotFoundError:
    print(f"'{TEST_PATH}' not found — upload it via the Files panel first.")
    test_df = None

if test_df is not None:
    rows, t0 = [], time.time()

    for i, row in test_df.iterrows():
        pred, all_preds = solve(row["problem"], n=SOLUTIONS_PER_PROBLEM, max_rounds=5)
        if pred is None:
            pred = 0
        votes = Counter(a for a in all_preds if a is not None)
        elapsed = time.time() - t0
        print(f"[{i+1:>3}/{len(test_df)}] id={row['id']}  answer={pred}  "
              f"votes={dict(votes.most_common(3))}  elapsed={elapsed:.0f}s")
        rows.append({"id": row["id"], "answer": int(pred)})

    sub_df = pd.DataFrame(rows)
    sub_df.to_csv(OUT_PATH, index=False)
    print(f"\nSaved {len(sub_df)} predictions → {OUT_PATH}")
    print(sub_df.to_string(index=False))

'test.csv' not found — upload it via the Files panel first.
